In [45]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [46]:
X, y = fetch_california_housing(return_X_y=True)
print(f"Dataset shapes: X: {X.shape}, y: {y.shape}")

Dataset shapes: X: (20640, 8), y: (20640,)


In [47]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [48]:
X_val, X_test, y_val, y_test = train_test_split(X, y, test_size=0.15)
X_train, X_blend, y_train, y_blend = train_test_split(X_val, y_val, test_size=0.2)
print(f"Train: {X_train.shape} | Blending: {X_blend.shape} | Test: {X_test.shape}")
print(f"""
Train: {len(X_train) / len(X):.2f} %
Blending: {len(X_blend) / len(X):.2f} %
Test: {len(X_test) / len(X):.2f} %
      """)

Train: (14035, 8) | Blending: (3509, 8) | Test: (3096, 8)

Train: 0.68 %
Blending: 0.17 %
Test: 0.15 %
      


In [49]:
!pip -q install lightgbm

In [50]:
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge

In [51]:
base_models = [
    {
        "name": "SVR",
        "model": SVR(C=5)
    },
    {
        "name": "KNN",
        "model": KNeighborsRegressor(
            n_neighbors=7,
            weights="distance",
            algorithm="brute",
            n_jobs=-1
        )
    },
    {
        "name": "RandomForest",
        "model": RandomForestRegressor(
            n_estimators=300,
            max_depth=5,
            min_samples_split=10,
            min_samples_leaf=5,
            n_jobs=-1
        )
    },
    {
        "name": "LightGBM",
        "model": LGBMRegressor(
            n_estimators=300,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=5,
            n_jobs=-1,
            verbose=0,
            force_col_wise=True
        )
    }
]

In [52]:
import time
train_times=[]
for item in base_models:
  start=time.time()
  item["model"].fit(X_train, y_train)
  print(item)
  end=time.time()
  train_times.append({item['name']: end-start})

{'name': 'SVR', 'model': SVR(C=5)}
{'name': 'KNN', 'model': KNeighborsRegressor(algorithm='brute', n_jobs=-1, n_neighbors=7,
                    weights='distance')}
{'name': 'RandomForest', 'model': RandomForestRegressor(max_depth=5, min_samples_leaf=5, min_samples_split=10,
                      n_estimators=300, n_jobs=-1)}
{'name': 'LightGBM', 'model': LGBMRegressor(colsample_bytree=0.8, force_col_wise=True, learning_rate=0.05,
              n_estimators=300, n_jobs=-1, reg_lambda=5, subsample=0.8,
              verbose=0)}


In [53]:
for train_info in train_times:
  print(train_info)

{'SVR': 22.918131351470947}
{'KNN': 0.0030684471130371094}
{'RandomForest': 15.744831323623657}
{'LightGBM': 0.5446608066558838}


In [54]:
import numpy as np
base_preds = [item["model"].predict(X_blend) for item in base_models]
# print(base_preds)
meta_X = np.column_stack(base_preds)
# print(meta_X)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [55]:
meta_model=Ridge()
meta_model.fit(meta_X, y_blend)

Ridge()

In [56]:
from sklearn.metrics import root_mean_squared_error, r2_score
def calc_errors(y_true, y_pred):
  return f"""
  RMSE: {root_mean_squared_error(y_true, y_pred)}
  R2: {r2_score(y_true, y_pred)} """

In [57]:
base_preds_test = [item["model"].predict(X_test) for item in base_models]
meta_X_test = np.column_stack(base_preds_test)
meta_pred = meta_model.predict(meta_X_test)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [58]:
print(calc_errors(y_test, meta_pred))


  RMSE: 0.43512665281212287
  R2: 0.856617360612473 


In [59]:
print(calc_errors(y_test, meta_pred))


  RMSE: 0.43512665281212287
  R2: 0.856617360612473 


In [60]:
for item in base_models:
  print(f"{item["name"]}: {calc_errors(y_test, item["model"].predict(X_test))}")

SVR: 
  RMSE: 0.5561526469395319
  R2: 0.7657642304605313 
KNN: 
  RMSE: 0.6190178405991144
  R2: 0.7098172911177998 
RandomForest: 
  RMSE: 0.6586935254628784
  R2: 0.6714269130601611 
LightGBM: 
  RMSE: 0.43827689949703574
  R2: 0.8545337108738326 


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
